In [ ]:
from pylab import *
import matplotlib.pyplot as plt
from matplotlib import ticker, cm
from matplotlib import colors
from scipy import integrate
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 unused import
from scipy import interpolate
%matplotlib inline

In [ ]:
import os

In [ ]:
os.chdir("/mnt/e/RBS/l=6/")

In [ ]:
os.mkdir("idata")

In [ ]:
os.listdir()

In [ ]:
w0 = 0.510
wf = 0.900
dw = 0.01

In [ ]:
ws = np.linspace(w0, wf, int((wf - w0) / dw) + 1)
print(ws, size(ws))
ws[18] = 0.6903

In [ ]:
l, dr, N = 6, 0.125, 512

In [ ]:
dirnames = []
for i in range(len(ws)):
    dirnames.insert(i, "l=%d,w=%0.5f,dr=%0.3f,N=%04d/" % (l, ws[i], dr, N))

In [ ]:
dz, order, ghost, m = dr, 4, 2, 1.0

In [ ]:
z_l = np.linspace(dz * (0.5 - ghost), dz * (N - 0.5 + ghost), N + 2 * ghost)
r_l = np.linspace(dz * (0.5 - ghost), dr * (N - 0.5 + ghost), N + 2 * ghost)
z, r = np.meshgrid(z_l, r_l)

In [ ]:
# Interpolating ranges.
i_dr, i_dz = 0.0625, 0.0625
i_N, i_ghost = 1024, 2
i_z_l = np.linspace(i_dz * (0.5 - i_ghost), i_dz * (i_N - 0.5 + i_ghost), i_N + 2 * i_ghost)
i_r_l = np.linspace(i_dz * (0.5 - i_ghost), i_dr * (i_N - 0.5 + i_ghost), i_N + 2 * i_ghost)

In [ ]:
for i in range(18, len(ws)):    
    # Read data.
    w = ws[i]
    log_alpha = np.genfromtxt(dirnames[i] + "log_alpha_f.asc")
    beta      = np.genfromtxt(dirnames[i] + "beta_f.asc")
    log_h     = np.genfromtxt(dirnames[i] + "log_h_f.asc")
    log_a     = np.genfromtxt(dirnames[i] + "log_a_f.asc")
    psi       = np.genfromtxt(dirnames[i] + "psi_f.asc")
    
    # Interpolating functions.
    if_log_alpha = interpolate.RectBivariateSpline(r_l, z_l, log_alpha)
    if_beta      = interpolate.RectBivariateSpline(r_l, z_l, beta)
    if_log_h     = interpolate.RectBivariateSpline(r_l, z_l, log_h)
    if_log_a     = interpolate.RectBivariateSpline(r_l, z_l, log_a)
    if_psi       = interpolate.RectBivariateSpline(r_l, z_l, psi)
    
    # Make directory.
    save_dir = "idata/l=%d,w=%0.6f,dr=%0.5f,N=%04d/" % (l, w, i_dr, i_N)
    os.mkdir(save_dir)
    
    # Save into files.
    np.savetxt(save_dir + "log_alpha_i.asc", if_log_alpha(i_r_l, i_z_l))
    np.savetxt(save_dir + "beta_i.asc", if_beta(i_r_l, i_z_l))
    np.savetxt(save_dir + "log_h_i.asc", if_log_h(i_r_l, i_z_l))
    np.savetxt(save_dir + "log_a_i.asc", if_log_a(i_r_l, i_z_l))
    np.savetxt(save_dir + "psi_i.asc", if_psi(i_r_l, i_z_l))

In [ ]:
# Global parameters.
global_dr, global_dz = i_dr, i_dz
global_NrInterior = global_NzInterior = i_N

In [ ]:
for i in range(ws.size):
    # Global parameters.
    par_order = 4
    par_l = l
    par_m = m
    par_w = ws[i]
    
    # Grid.
    par_dr = global_dr
    par_dz = global_dz
    par_NrInterior = global_NrInterior
    par_NzInterior = global_NzInterior
    
    # Initial data read.
    par_readInitialData = 1
    read_dr = global_dr
    read_N = global_NrInterior
    
    read_dir = "../idata/l=%d,w=%.6f,dr=%.5f,N=%04d/" % (par_l, par_w, par_dr, par_NrInterior)
    read_prefix = ""
    read_suffix = "_i"
        
    par_NrTotalInitial = read_N + par_order
    par_NzTotalInitial = read_N + par_order
    par_log_alpha_i = read_dir + read_prefix + "log_alpha" + read_suffix + ".asc"
    par_beta_i = read_dir + read_prefix + "beta" + read_suffix + ".asc"
    par_log_h_i = read_dir + read_prefix + "log_h" + read_suffix + ".asc"
    par_log_a_i = read_dir + read_prefix + "log_a" + read_suffix + ".asc"
    par_psi_i = read_dir + read_prefix + "psi" + read_suffix + ".asc"
    
    # Output directory.
    par_dirname = "l=%d,w=%.6f,dr=%.5f,N=%04d" % (par_l, par_w, par_dr, par_NrInterior)
    
    par_grid = """# GRID
dr = %18.16E
dz = %18.16E
NrInterior = %d
NzInterior = %d
order = %d

""" % (par_dr, par_dz, par_NrInterior, par_NzInterior, par_order)

    par_scalar_field_props = """# SCALAR FIELD PROPERTIES 
l = %d
m = %18.16E
w0 = %18.16E

""" % (par_l, par_m, par_w)

    par_initial_data = """# INITIAL DATA
readInitialData = %d
NrTotalInitial = %d
NzTotalInitial = %d
log_alpha_i\t= \"%s\"
beta_i\t\t= \"%s\"
log_h_i\t\t= \"%s\"
log_a_i\t\t= \"%s\"
psi_i\t\t= \"%s\"

""" % (par_readInitialData, par_NrTotalInitial, par_NzTotalInitial,
      par_log_alpha_i, par_beta_i, par_log_h_i, par_log_a_i, par_psi_i)

    par_output = """# OUTPUT DIRECTORY
dirname = \"%s\"

""" % (par_dirname)

    par_no_touch = """# DO NOT TOUCH THESE PARAMETERS!
#SOLVER PARAMETERS
solverType = 1
localSolver = 1
epsilon = 1.0E-12
maxNewtonIter = 10
lambda0 = 1.0E-4
lambdaMin = 1.0E-6
useLowRank = 1

# ANALYTIC INITIAL DATA AND RESCALE.
psi0 = 1.0

# FIXED VARIABLE.
fixedPhi = 0
fixedPhiR = 0
fixedPhiZ = 0
fixedOmega = 1

#BOUNDARY TYPES
alphaBoundOrder = 1
betaBoundOrder = 1
hBoundOrder = 1
aBoundOrder = 1
phiBoundOrder = 1

"""
    
    # Generate parfile.
    parfile = par_grid + par_scalar_field_props + par_initial_data + par_output + par_no_touch
    
    # Open file.
    file = open("idata/l=%d,w=%.6f,dr=%.5f,N=%04d.par" % (par_l, par_w, par_dr, par_NrInterior), "w")
    file.write(parfile)
    file.close()

In [ ]:
pbs_exe = """#!/bin/bash
#
# Name of job.
#PBS -N ROTBOSON
#
# Output files.
#PBS -o $PBS_JOBNAME.$PBS_JOBID.out
#PBS -e $PBS_JOBNAME.$PBS_JOBID.err
#
# Set to "mpi" queue.
#PBS -q mpi
#
# Resources.
#PBS -l nodes=1:ppn=16
#PBS -l mem=90gb
#PBS -l vmem=110gb
#
# Walltime.
#PBS -l walltime=48:00:00
#
# Email notifications.
#PBS -m abe -M santiago.ontanon@correo.nucleares.unam.mx

# Change to current directory.
cd $PBS_O_WORKDIR

# Send email indicating job start.
echo -e "Subject: $PBS_JOBNAME.$PBS_JOBID \\n\\nExecution has begun." |  sendmail santiago.ontanon@correo.nucleares.unam.mx

# Job information.
echo ==============================
echo Ejecutandose en: `hostname`
echo Fecha: `date`
echo Directorio: `pwd`
echo Recursos asignados:
echo 	`cat $PBS_NODEFILE`
NPROCS=`wc -l < $PBS_NODEFILE`
echo Total: $NPROCS cpus
echo ==============================
cat $PBS_NODEFILE > $HOME/nodos
echo ==============================
echo 		SALIDA
echo ==============================

# Set OMP_NUM_THREADS.
export OMP_NUM_THREADS=$PBS_NUM_PPN
export MKL_NUM_THREASD=$PBS_NUM_PPN

# Job starts
"""
for i in range(ws.size):
    pbs_exe += """time ./ROTBOSON \"idata/l=%d,w=%.6f,dr=%.5f,N=%04d.par\"
echo -e "Subject: $PBS_JOBNAME.$PBS_JOBID \\n\\nRun %03d Finished." |  sendmail santiago.ontanon@correo.nucleares.unam.mx
""" % (l, ws[i], i_dr, i_N, i + 1)
    
pbs_exe += """
# Job ends.
echo ==============================
echo -e "Subject: $PBS_JOBNAME.$PBS_JOBID \\n\\nAll execution done!" |  sendmail santiago.ontanon@correo.nucleares.unam.mx
"""

In [ ]:
# Open file.
file = open("idata/torque_exe_script.pbs", "w")
file.write(pbs_exe)
file.close()